In [1]:
import numpy as np
import pandas as pd

import altair as alt
alt.data_transformers.enable("vegafusion")
import bokeh.io
bokeh.io.output_notebook()
import iqplot

import opendp.prelude as dp

/Users/rf50/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loading BokehJS ...

In [2]:
block_df = pd.read_csv("data/processed_data/DF_IL_2010_BLOCK.csv")
tract_df = pd.read_csv("data/processed_data/DF_IL_2010_TRACT.csv")
county_df = pd.read_csv("data/processed_data/DF_IL_2010_COUNTY.csv")
state_df = pd.read_csv("data/processed_data/DF_IL_2010_STATE.csv")

# Add noise

In [20]:
dp.enable_features("contrib")

RACE_COLS = ["white", "black", "asian", "other"]

def add_opendp_laplace_noise(df, epsilon, cols=RACE_COLS):
    """
    Toy block-level DP noising for race cells.
    Sensitivity = 1 per cell query, scale = 1 / epsilon.
    """
    out = df.copy()
    scale = 1.0 / epsilon

    meas = dp.m.make_laplace(
        dp.atom_domain(T=float, nan=False),
        dp.absolute_distance(T=float),
        scale=scale,
    )

    for col in cols:
        out[f"noisy_{col}"] = out[col].astype(float).map(lambda x: meas(x))

    out["noisy_pop"] = out[[f"noisy_{c}" for c in cols]].sum(axis=1)
    out["epsilon"] = epsilon
    return out

In [21]:
block_df = block_df.replace([np.inf, -np.inf], np.nan)
block_df = block_df.fillna(0)

In [23]:
block_df_noised = add_opendp_laplace_noise(block_df, epsilon=0.2)

In [24]:
block_df_noised.head()

,geoid,parent_geoid,true_pop,white,black,asian,other,noisy_white,noisy_black,noisy_asian,noisy_other,noisy_pop,epsilon
0,4009,10200,28,25,0,1,2,21.829365,-0.135374,-2.773377,-10.060244,8.860370,0.2
1,4010,10200,8,8,0,0,0,-3.317955,-4.213580,-6.808800,7.066869,-7.273466,0.2
2,4011,10200,8,8,0,0,0,8.645114,0.106013,-8.737957,7.392715,7.405885,0.2
3,4012,10200,16,16,0,0,0,15.173057,3.549002,-1.332341,-4.462213,12.927504,0.2
4,4013,10200,37,37,0,0,0,40.524974,1.014084,-0.289022,-5.965062,35.284974,0.2


In [ ]:
block_df_noised.to_csv('data/processed_data/DP/DF_IL_2010_DP.csv')

In [18]:
print(block_df[RACE_COLS].isna().sum())
print(np.isfinite(block_df[RACE_COLS].to_numpy()).all())

white    0
black    0
asian    0
other    0
dtype: int64
True


In [ ]:
# simple post-processing
def postprocess_block_race_counts(df, cols=RACE_COLS):
    """
    Simple toy postprocessing:
    1. clip race counts at 0
    2. round to integers
    3. recompute population as sum of race cells
    """
    out = df.copy()

    for col in cols:
        noisy_col = f"noisy_{col}"
        adj_col = f"adj_{col}"
        out[adj_col] = out[noisy_col].clip(lower=0).round().astype(int)

    out["adj_pop"] = out[[f"adj_{c}" for c in cols]].sum(axis=1)
    return out

In [ ]:
df_noisy = add_opendp_laplace_noise(df_block, epsilon=1.0)
df_adj = postprocess_block_race_counts(df_noisy)

df_adj[[
    "geoid", "parent_geoid", "true_pop",
    "white", "black", "asian", "other",
    "adj_white", "adj_black", "adj_asian", "adj_other", "adj_pop"
]].head()

In [109]:
# row-independent noise
def add_laplace_noise(df, true_col, epsilon_level, seed=1273987):
    """
    Add Laplace noise with sensitivity 1.
    """
    rng = np.random.default_rng(seed)
    scale = 1.0 / epsilon_level
    noisy = df[true_col].to_numpy(dtype=float) + rng.laplace(0.0, scale, len(df))
    out = df.copy()
    out["noisy_pop"] = noisy
    out["epsilon_level"] = epsilon_level
    out["laplace_scale"] = scale
    return out

In [110]:
add_laplace_noise(block_df, 'white', epsilon_level=0.2)

,geoid,parent_geoid,true_pop,white,black,asian,other,noisy_pop,epsilon_level,laplace_scale
0,4009,10200,28,25,0,1,2,30.671859,0.2,5.0
1,4010,10200,8,8,0,0,0,11.547364,0.2,5.0
2,4011,10200,8,8,0,0,0,10.690390,0.2,5.0
3,4012,10200,16,16,0,0,0,2.718102,0.2,5.0
4,4013,10200,37,37,0,0,0,40.364980,0.2,5.0
...,...,...,...,...,...,...,...,...,...,...
451549,3021,30400,32,32,0,0,0,38.393341,0.2,5.0
451550,3022,30400,4,4,0,0,0,5.155974,0.2,5.0
451551,3023,30400,0,0,0,0,0,5.203776,0.2,5.0
451552,3024,30400,3,3,0,0,0,-5.736582,0.2,5.0


Simulate partitions

In [ ]:
## IGnore GPT code

In [66]:
# def largest_remainder_round(values, target_total):
#     """
#     Round nonnegative floats to integers that sum exactly to target_total.
#     """
#     values = np.asarray(values, dtype=float)
#     values = np.maximum(values, 0.0)

#     floors = np.floor(values).astype(int)
#     remainder = int(target_total - floors.sum())

#     if remainder > 0:
#         frac = values - floors
#         order = np.argsort(-frac)  # largest fractional parts first
#         floors[order[:remainder]] += 1
#     elif remainder < 0:
#         frac = values - floors
#         order = np.argsort(frac)  # smallest fractional parts lose 1 first
#         for idx in order[: -remainder]:
#             if floors[idx] > 0:
#                 floors[idx] -= 1

#     return floors


# def reconcile_group(child_df, parent_total, noisy_col="noisy_pop"):
#     """
#     Take one parent's children, clip to nonnegative, scale to parent total,
#     then integer-round while preserving the exact parent total.
#     """
#     x = child_df[noisy_col].to_numpy(dtype=float)
#     x = np.maximum(x, 0.0)

#     if parent_total <= 0:
#         child_df = child_df.copy()
#         child_df["adjusted_pop"] = 0
#         return child_df

#     s = x.sum()
#     if s == 0:
#         # Simple fallback for the demo: distribute uniformly
#         scaled = np.full(len(x), parent_total / len(x))
#     else:
#         scaled = x * (parent_total / s)

#     child_df = child_df.copy()
#     child_df["adjusted_pop"] = largest_remainder_round(scaled, int(parent_total))
#     return child_df


# def add_laplace_noise(df, true_col, epsilon_level, rng):
#     """
#     Add Laplace noise with sensitivity 1.
#     """
#     scale = 1.0 / epsilon_level
#     noisy = df[true_col].to_numpy(dtype=float) + rng.laplace(0.0, scale, len(df))
#     out = df.copy()
#     out["noisy_pop"] = noisy
#     out["epsilon_level"] = epsilon_level
#     out["laplace_scale"] = scale
#     return out


# def simulate_topdown(
#     state_df,
#     county_df,
#     tract_df,
#     block_df,
#     epsilon_total=1.0,
#     weights=None,
#     invariant_state=True,
#     seed=0,
# ):
#     """
#     Expected columns:
#       state_df:  geoid, true_pop
#       county_df: geoid, parent_geoid, true_pop
#       tract_df:  geoid, parent_geoid, true_pop
#       block_df:  geoid, parent_geoid, true_pop

#     parent_geoid should point to the parent's geoid at the next higher level.
#     Assumes a single state for the demo.
#     """
#     if weights is None:
#         weights = {
#             "state": 0.05,
#             "county": 0.15,
#             "tract": 0.30,
#             "block": 0.50,
#         }

#     if not np.isclose(sum(weights.values()), 1.0):
#         raise ValueError("weights must sum to 1.0")

#     rng = np.random.default_rng(seed)

#     # 1. Noisy measurements at every level
#     state = state_df.copy()
#     if invariant_state:
#         state["noisy_pop"] = state["true_pop"].astype(float)
#         state["epsilon_level"] = np.inf
#         state["laplace_scale"] = 0.0
#     else:
#         state = add_laplace_noise(state, "true_pop", epsilon_total * weights["state"], rng)

#     county = add_laplace_noise(county_df, "true_pop", epsilon_total * weights["county"], rng)
#     tract = add_laplace_noise(tract_df, "true_pop", epsilon_total * weights["tract"], rng)
#     block = add_laplace_noise(block_df, "true_pop", epsilon_total * weights["block"], rng)

#     # 2. Reconcile counties to state
#     state_total = int(state.iloc[0]["true_pop"] if invariant_state else max(0, round(state.iloc[0]["noisy_pop"])))
#     state["adjusted_pop"] = state_total

#     county = reconcile_group(county, parent_total=state_total, noisy_col="noisy_pop")

#     # 3. Reconcile tracts to each county
#     tract_parts = []
#     county_totals = county.set_index("geoid")["adjusted_pop"].to_dict()
#     for county_geoid, g in tract.groupby("parent_geoid", sort=False):
#         parent_total = int(county_totals[county_geoid])
#         tract_parts.append(reconcile_group(g, parent_total=parent_total, noisy_col="noisy_pop"))
#     tract = pd.concat(tract_parts, ignore_index=True)

#     # 4. Reconcile blocks to each tract
#     block_parts = []
#     tract_totals = tract.set_index("geoid")["adjusted_pop"].to_dict()
#     for tract_geoid, g in block.groupby("parent_geoid", sort=False):
#         parent_total = int(tract_totals[tract_geoid])
#         block_parts.append(reconcile_group(g, parent_total=parent_total, noisy_col="noisy_pop"))
#     block = pd.concat(block_parts, ignore_index=True)

#     return {
#         "state": state,
#         "county": county,
#         "tract": tract,
#         "block": block,
#     }


In [ ]:
# result = simulate_topdown(
#     state_df,
#     df_il_county,
#     df_il_tract,
#     df_il_block,
#     epsilon_total=1.0,
#     invariant_state=True,
#     seed=42,
# )

# result["block"][["geoid", "true_pop", "noisy_pop", "adjusted_pop"]].head()